In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
X,y = load_iris(return_X_y=True)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
rf = RandomForestClassifier(n_estimators=100,random_state=42)

In [7]:
rf.fit(X_train,y_train)

RandomForestClassifier(random_state=42)

In [8]:
y_pred = rf.predict(X_test)

In [9]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 1.0


In [ ]:
# rf = RandomForestClassifier(
#     n_estimators=100,
#     max_depth=5,
#     min_samples_split=2
# )
# rf.fit(X_train, y_train)
# y_pred = rf.predict(X_test)

In [10]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np

In [11]:
# Create synthetic data
X, y = make_classification(
    n_samples=500,
    n_features=5,
    n_informative=3,
    n_redundant=0,
    random_state=42
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
np.random.seed(42)

indices = np.random.choice(len(X_train), size=len(X_train), replace=True)

print("First 15 sampled indices:", indices[:15])
print("Unique samples:", len(np.unique(indices)))
print("Total samples drawn:", len(indices))

First 15 sampled indices: [102 348 270 106  71 188  20 102 121 214 330  87 372  99 359]
Unique samples: 238
Total samples drawn: 400


In [13]:
rf = RandomForestClassifier(
    n_estimators=50,
    bootstrap=True,      # sampling with replacement
    oob_score=True,      # enable OOB error estimation
    random_state=42
)

rf.fit(X_train, y_train)


RandomForestClassifier(n_estimators=50, oob_score=True, random_state=42)

In [14]:
# Predictions on test set
y_pred = rf.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
oob_acc = rf.oob_score_

print("Test Accuracy:", test_acc)
print("OOB Accuracy :", oob_acc)


Test Accuracy: 0.94
OOB Accuracy : 0.9125


In [15]:
oob_errors = []

for n in [5, 10, 20, 50, 100]:
    rf = RandomForestClassifier(
        n_estimators=n,
        bootstrap=True,
        oob_score=True,
        random_state=42
    )
    rf.fit(X_train, y_train)
    oob_error = 1 - rf.oob_score_
    oob_errors.append(oob_error)
    print(f"Trees: {n}, OOB Error: {oob_error:.3f}")


/usr/local/lib/python3.12/dist-packages/sklearn/ensemble/_forest.py:612: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(


Trees: 5, OOB Error: 0.185
Trees: 10, OOB Error: 0.108
Trees: 20, OOB Error: 0.093
Trees: 50, OOB Error: 0.088
Trees: 100, OOB Error: 0.080


In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
np.random.seed(42)

print("="*80)
print("RANDOM FOREST CLASSIFIER - COMPREHENSIVE TUTORIAL")
print("="*80)

RANDOM FOREST CLASSIFIER - COMPREHENSIVE TUTORIAL


In [17]:
print("\n" + "="*80)
print("SECTION 1: DATASET INFORMATION")
print("="*80)

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"\nDataset: Iris Dataset")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Features: {feature_names}")
print(f"Target classes: {target_names}")
print(f"Class distribution: {np.bincount(y)}")

df = pd.DataFrame(X, columns=feature_names)
df['species'] = pd.Categorical.from_codes(y, target_names)

print("\nFirst 10 rows:")
print(df.head(10))

print("\nDataset Statistics:")
print(df.describe())

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")


SECTION 1: DATASET INFORMATION

Dataset: Iris Dataset
Number of samples: 150
Number of features: 4
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target classes: ['setosa' 'versicolor' 'virginica']
Class distribution: [50 50 50]

First 10 rows:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   
5                5.4               3.9                1.7               0.4   
6                4.6               3.4                1.4               0.3   
7                5.0               3.4                1.5               0.2 

In [18]:
print("\n" + "="*80)
print("SECTION 2: ENTROPY AND INFORMATION GAIN")
print("="*80)

def calculate_entropy(y):
    """Calculate entropy of a label distribution"""
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / counts.sum()
    entropy = -np.sum(probabilities * np.log2(probabilities + 1e-10))
    return entropy

def calculate_information_gain(X_column, y, threshold):
    """Calculate information gain for a split"""
    parent_entropy = calculate_entropy(y)

    left_mask = X_column <= threshold
    right_mask = X_column > threshold

    if left_mask.sum() == 0 or right_mask.sum() == 0:
        return 0

    n = len(y)
    n_left = left_mask.sum()
    n_right = right_mask.sum()

    left_entropy = calculate_entropy(y[left_mask])
    right_entropy = calculate_entropy(y[right_mask])

    weighted_child_entropy = (n_left/n) * left_entropy + (n_right/n) * right_entropy
    information_gain = parent_entropy - weighted_child_entropy

    return information_gain

# Example calculation
feature_idx = 2  # petal length
feature_values = X_train[:, feature_idx]
threshold = np.median(feature_values)

print(f"\nExample: Information Gain for '{feature_names[feature_idx]}'")
print(f"Threshold: {threshold:.4f}")

parent_entropy = calculate_entropy(y_train)
print(f"\nParent Entropy: {parent_entropy:.4f}")

unique, counts = np.unique(y_train, return_counts=True)
print("\nClass distribution:")
for cls, count in zip(unique, counts):
    prob = count / len(y_train)
    print(f"  {target_names[cls]}: {count} samples (p={prob:.4f})")

print("\nEntropy calculation:")
print("H(S) = -Σ(p_i * log2(p_i))")
for cls, count in zip(unique, counts):
    prob = count / len(y_train)
    contribution = -prob * np.log2(prob)
    print(f"  -{prob:.4f} * log2({prob:.4f}) = {contribution:.4f}")
print(f"  Total: {parent_entropy:.4f}")

# Calculate split
left_mask = feature_values <= threshold
right_mask = feature_values > threshold

print(f"\nAfter split at {threshold:.4f}:")
print(f"  Left child: {left_mask.sum()} samples")
print(f"  Right child: {right_mask.sum()} samples")

left_entropy = calculate_entropy(y_train[left_mask])
right_entropy = calculate_entropy(y_train[right_mask])

print(f"\nLeft entropy: {left_entropy:.4f}")
print(f"Right entropy: {right_entropy:.4f}")

weighted_entropy = (left_mask.sum()/len(y_train)) * left_entropy + \
                   (right_mask.sum()/len(y_train)) * right_entropy

info_gain = parent_entropy - weighted_entropy
print(f"\nWeighted child entropy: {weighted_entropy:.4f}")
print(f"Information Gain: {info_gain:.4f}")


SECTION 2: ENTROPY AND INFORMATION GAIN

Example: Information Gain for 'petal length (cm)'
Threshold: 4.3000

Parent Entropy: 1.5802

Class distribution:
  setosa: 31 samples (p=0.2952)
  versicolor: 37 samples (p=0.3524)
  virginica: 37 samples (p=0.3524)

Entropy calculation:
H(S) = -Σ(p_i * log2(p_i))
  -0.2952 * log2(0.2952) = 0.5196
  -0.3524 * log2(0.3524) = 0.5303
  -0.3524 * log2(0.3524) = 0.5303
  Total: 1.5802

After split at 4.3000:
  Left child: 53 samples
  Right child: 52 samples

Left entropy: 0.9791
Right entropy: 0.8667

Weighted child entropy: 0.9234
Information Gain: 0.6567
